<a href="https://colab.research.google.com/github/kinemax-core/Genomic-data-science-R/blob/main/dna_edit_dist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request

# 1. Download the human chromosome 1 excerpt
url = "https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/chr1.GRCh38.excerpt.fasta"
filename = "chr1.GRCh38.excerpt.fasta"
urllib.request.urlretrieve(url, filename)

# 2. Parse FASTA file
def readGenome(filename):
    genome = ''
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                genome += line.rstrip()
    return genome

t = readGenome(filename)

# 3. Edit distance for approximate matching
def approximate_match_edit_distance(p, t):
    len_p = len(p)
    len_t = len(t)

    matrix = [[0] * (len_t + 1) for _ in range(len_p + 1)]

    for i in range(len_p + 1):
        matrix[i][0] = i

    for i in range(1, len_p + 1):
        for j in range(1, len_t + 1):
            cost = 0 if p[i-1] == t[j-1] else 1
            matrix[i][j] = min(
                matrix[i-1][j] + 1,
                matrix[i][j-1] + 1,
                matrix[i-1][j-1] + cost
            )

    return min(matrix[len_p])

# --- Sanity Check ---
test_p = "GCGTATGC"
test_t = "TATTGGCTATACGGTT"
print(f"Sanity Check: {approximate_match_edit_distance(test_p, test_t)} (Expected: 2)")

# --- Main Query ---
p = "GATTTACCAGATTGAG"
result = approximate_match_edit_distance(p, t)
print(f"Edit distance of closest match: {result}")

In [ ]:
import urllib.request
from collections import defaultdict

# 1. Download and parse FASTQ
def readFastq(filename):
    sequences = []
    with open(filename) as f:
        while True:
            f.readline()          # read name (@...)
            seq = f.readline().rstrip()   # sequence
            f.readline()          # +
            f.readline()          # quality
            if not seq:
                break
            sequences.append(seq)
    return sequences

url = "https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/ERR266411_1.for_asm.fastq"
urllib.request.urlretrieve(url, "reads.fastq")
reads = readFastq("reads.fastq")

# 2. Overlap function
def overlap(a, b, min_length=3):
    """Return length of longest overlap where suffix of a matches prefix of b."""
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1

# 3. Build k-mer index: kmer -> set of reads containing it
def build_kmer_index(reads, k):
    index = defaultdict(set)
    for read in reads:
        for i in range(len(read) - k + 1):
            index[read[i:i+k]].add(read)
    return index

# 4. Find all overlapping pairs efficiently
def overlap_all_pairs(reads, k):
    index = build_kmer_index(reads, k)
    reads_set = set(reads)
    overlap_pairs = set()

    for a in reads:
        suffix_kmer = a[-k:]  # length-k suffix of a
        candidates = index[suffix_kmer]  # reads containing that k-mer
        for b in candidates:
            if a != b:  # don't overlap with itself
                olen = overlap(a, b, min_length=k)
                if olen > 0:
                    overlap_pairs.add((a, b))

    return overlap_pairs

k = 30
pairs = overlap_all_pairs(reads, k)

print(f"Number of overlapping pairs: {len(pairs)}")

# Count unique reads that have at least one overlap as a left read
left_reads = set(a for a, b in pairs)
print(f"Number of reads with a suffix overlap: {len(left_reads)}")